# Embeddings and Relevant Search - Concepts

**Business problem:** find useful job ads early in a ranked list.

A model supports a decision; it does not make the business decision for us.

## Learning objectives

- Distinguish ranking ads from predicting categories.
- Explain independent text encoding and rank fusion.
- Interpret Precision@K, Recall@K and first relevant rank.

## Dataset and practical scope

The supplied ranked lists below are constructed. The exercise uses the 24 approved excerpts and their draft relevance judgments.

All code is supplied. Run the cells from top to bottom.

The companion [exercise](embedding_search_exercise.ipynb) and [solution](embedding_search_solution.ipynb) first evaluate 24 approved excerpts, then demonstrate interactive search over the 5,011 real ads from Lecture 2. The larger collection has no full-ad relevance judgments; its results are inspected, not scored using the excerpt labels.

## Libraries and settings

Use the prepared Python 3.11 Codespace kernel. No GPU or API key is needed.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SEED = 7
np.random.seed(SEED)
DATA_ROOT = Path('../../../00_Data')

## 1. Establish the search goal and baseline

The user types a query. Relevance concerns a query-ad pair, not one permanent category. BM25 is a lexical baseline using term counts, rarity and length. Exact skills can matter; matching words need not establish the right work.

## 2. Retrieval-trained embeddings

Encode ads once and cache their vectors. Encode each new query and compare vectors. Pooling plain BERT permits caching, but does not train useful cosine geometry. E5 is a finished multilingual retrieval encoder used through the Sentence Transformers library.

In [2]:
query_vector = np.array([1., 0.])
ad_vectors = np.array([[0.9, 0.1], [0.2, 0.8], [0.7, 0.3]])
ad_vectors /= np.linalg.norm(ad_vectors, axis=1, keepdims=True)
display(pd.DataFrame({'ad': ['A', 'B', 'C'],
                      'constructed cosine': ad_vectors @ query_vector})
        .sort_values('constructed cosine', ascending=False))

,ad,constructed cosine
0,A,0.993884
2,C,0.919145
1,B,0.242536


## 3. Combine lists, not raw score scales

Reciprocal Rank Fusion (RRF) sums 1/(60 + rank) for each result across lists. Rank begins at 1. A missing result contributes zero. Combining signals can reinforce an error.

In [3]:
supplied_lists = [['A', 'B', 'C'], ['A', 'C', 'D']]
votes = {}
for ranking in supplied_lists:
    for rank, ad in enumerate(ranking, 1):
        votes[ad] = votes.get(ad, 0) + 1 / (60 + rank)
display(pd.Series(votes, name='RRF score').sort_values(ascending=False).to_frame())

,RRF score
A,0.032787
C,0.032002
B,0.016129
D,0.015873


## 4. Evaluate a useful top three

Assume only B and D are relevant, and the returned order is A, B, C. Precision counts useful slots; recall counts coverage; reciprocal rank uses the first relevant position. Mean reciprocal rank averages across queries.

In [4]:
ranking, relevant, k = ['A', 'B', 'C'], {'B', 'D'}, 3
positions = [rank for rank, ad in enumerate(ranking[:k], 1) if ad in relevant]
display(pd.Series({'Precision@3': len(positions) / k,
                   'Recall@3': len(positions) / len(relevant),
                   'RR@3': 1 / positions[0] if positions else 0.}))

Precision@3    0.333333
Recall@3       0.500000
RR@3           0.500000
dtype: float64

## Interpretation

Equal first-hit rank need not mean equal coverage. Scores are not relevance labels. Define the intent before evaluating and verify requirements such as location or shifts outside the similarity score.

## Summary and business implications

Select a recipe from evidence, not because it has more components. Validate user usefulness, response time and cost separately from offline retrieval metrics.

## Limitations

This is a small teaching example, not evidence of deployment performance. Keep evaluation separate from fitting and model selection. Check errors and the user's decision, not only a headline metric.

## Further exercises

- Can fusion recover an ad missing from both candidate lists?
- Why can a high cosine score still fail a location requirement?
- Is a click an unbiased relevance label?

## References

**Required reading:** https://www.sbert.net/examples/applications/semantic-search/README.html
and https://huggingface.co/intfloat/multilingual-e5-small
**Instructor references:** Reimers & Gurevych (2019), *Sentence-BERT*, https://aclanthology.org/D19-1410/;
Wang et al. (2024), *Multilingual E5 Text Embeddings*, https://arxiv.org/abs/2402.05672;
Cormack et al. (2009), *Reciprocal rank fusion*, https://doi.org/10.1145/1571941.1572114.
**Official BM25 documentation:** https://www.elastic.co/docs/reference/elasticsearch/index-settings/similarity

### Jupyter notebook - footer info

Include your name, submission date and a short description of your changes when submitting.